# Adding Background to Lab Images

Composites lab (white-background) leaves onto real field backgrounds: segment the leaf (RMBG-1.4), crop and lightly blur a background patch, alpha-paste the leaf, and match brightness. Includes UMAP, per-channel colour statistics and FID checks comparing composited vs. real-field images.

## Setup

In [ ]:
!pip install "numpy>2.0.0" pytorch-fid transformers pillow

## Compositing pipeline

In [ ]:
import cv2
import numpy as np
import os
import random
from tqdm import tqdm
from transformers import pipeline
from transformers.modeling_utils import PreTrainedModel
from PIL import Image as PILImage

if not hasattr(PreTrainedModel, 'all_tied_weights_keys'):
    PreTrainedModel.all_tied_weights_keys = {}

segmenter = pipeline("image-segmentation", 
                     model="briaai/RMBG-1.4", 
                     trust_remote_code=True)

def segment_leaf(leaf_img):
    img_rgb = cv2.cvtColor(leaf_img, cv2.COLOR_BGR2RGB)
    pil_img = PILImage.fromarray(img_rgb)
    
    result  = segmenter(pil_img)
    
    mask    = np.array(result)
    
    if mask.ndim == 3 and mask.shape[2] == 4:
        mask = mask[:, :, 3]
    elif mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
    
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        clean_mask = np.zeros_like(mask)
        largest    = max(contours, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [largest], -1, 255, thickness=cv2.FILLED)
        mask = clean_mask

    coverage = np.sum(mask > 0) / mask.size
    return mask, coverage

def crop_background(bg_img, target_size=(224, 224)):
    h, w = bg_img.shape[:2]
    th, tw = target_size

    crop_h = max(int(h * 0.7), th)
    crop_w = max(int(w * 0.7), tw)
    crop_h = min(crop_h, h)
    crop_w = min(crop_w, w)

    top  = random.randint(0, max(0, h - crop_h))
    left = random.randint(0, max(0, w - crop_w))
    
    patch = bg_img[top:top+crop_h, left:left+crop_w]
    patch = cv2.resize(patch, (tw, th), interpolation=cv2.INTER_LANCZOS4)
    return patch

def blur_background(bg_patch):
    return cv2.GaussianBlur(bg_patch, (9, 9), 2.5)

def paste_leaf(leaf_img, bg_patch, mask):
    h, w  = bg_patch.shape[:2]
    leaf_resized = cv2.resize(leaf_img, (w, h), interpolation=cv2.INTER_LANCZOS4)
    mask_resized = cv2.resize(mask,     (w, h), interpolation=cv2.INTER_NEAREST)
    
    mask_f   = cv2.GaussianBlur(mask_resized, (3, 3), 0.5)
    mask_f   = mask_f.astype(np.float32) / 255.0
    mask_3ch = np.stack([mask_f] * 3, axis=-1)

    leaf_f  = leaf_resized.astype(np.float32)
    bg_f    = bg_patch.astype(np.float32)
    result  = leaf_f * mask_3ch + bg_f * (1 - mask_3ch)
    return result.astype(np.uint8), mask_resized

def match_brightness(composited, bg_patch, mask):
    result = composited.astype(np.float32)
    
    bg_gray   = cv2.cvtColor(bg_patch, cv2.COLOR_BGR2GRAY).astype(np.float32)
    bg_mean   = np.mean(bg_gray)
    
    leaf_gray = cv2.cvtColor(composited, cv2.COLOR_BGR2GRAY).astype(np.float32)
    leaf_mean = np.mean(leaf_gray[mask > 0]) if np.any(mask > 0) else 1.0
    
    if leaf_mean > 0:
        factor = 1.0 + 0.3 * (bg_mean - leaf_mean) / leaf_mean
        factor = np.clip(factor, 0.8, 1.4)
        
        for c in range(3):
            ch = result[:, :, c]
            ch[mask > 0] = np.clip(ch[mask > 0] * factor, 0, 255)
            result[:, :, c] = ch

    return result.astype(np.uint8)

def process(leaf_path, bg_path):
    leaf_img = cv2.imread(leaf_path)
    bg_img   = cv2.imread(bg_path)

    if leaf_img is None or bg_img is None:
        return None

    mask, coverage = segment_leaf(leaf_img)
    if coverage <= 0.03 or coverage > 0.95:
        print(f"   Bad mask ({coverage:.2f}) — {os.path.basename(leaf_path)}")
        return None

    bg_patch = crop_background(bg_img)
    bg_patch = blur_background(bg_patch)

    composited, mask_resized = paste_leaf(leaf_img, bg_patch, mask)
    final = match_brightness(composited, bg_patch, mask_resized)
    return final

MSLDS = {
    "Healthy Leaf": r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300\Healthy Leaf",
}
BACKGROUND_DIRS = [
    r"D:\Desktop\Sesame Datasets\Combining Datasets\RSFD_Discarded_Images",
]
OUTPUT_ROOT  = r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300_300_Background"
NUM_PER_LEAF = 1

bg_pool = []
for d in BACKGROUND_DIRS:
    if not os.path.exists(d):
        print(f"  Missing: {d}")
        continue
    for f in os.listdir(d):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            bg_pool.append(os.path.join(d, f))

random.shuffle(bg_pool)
bg_index = 0
print(f"Background pool: {len(bg_pool)} images")

for class_name, MSLD_dir in MSLD_DIRS.items():
    output_dir = os.path.join(OUTPUT_ROOT, class_name)
    os.makedirs(output_dir, exist_ok=True)

    leaf_files = [
        os.path.join(MSLD_dir, f)
        for f in os.listdir(MSLD_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    processed, skipped = 0, 0
    print(f"\n{class_name} — {len(leaf_files)} leaves")

    for leaf_path in tqdm(leaf_files, desc=class_name):
        for i in range(NUM_PER_LEAF):
            bg_path = bg_pool[bg_index % len(bg_pool)]
            bg_index += 1

            result = process(leaf_path, bg_path)
            if result is None:
                skipped += 1
                continue

            out_name = f"{os.path.splitext(os.path.basename(leaf_path))[0]}_bg{i+1}.jpg"
            cv2.imwrite(os.path.join(output_dir, out_name), result,
                       [cv2.IMWRITE_JPEG_QUALITY, 95])
            processed += 1

    print(f"  {processed} saved | {skipped} skipped")

print(f"\nDone — review at {OUTPUT_ROOT}")

## UMAP validation (real vs composited)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import umap
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os

REAL_FIELD_DIR    = r"D:\Desktop\Sesame Datasets\Combining Datasets\RSFD300_160\Bacterial Blight Infected"
REAL_MSLD_DIR = r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300_300\Bacterial Blight Infected"
COMPOSITED_DIR    = r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300_300_Background\Bacterial Blight Infected"

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

backbone = models.resnet50(weights='IMAGENET1K_V1')
backbone.fc = torch.nn.Identity()
backbone = backbone.to(DEVICE).eval()

def extract_features_from_dir(folder, label, max_imgs=300):
    features, labels, paths = [], [], []
    files = [f for f in os.listdir(folder) 
             if f.lower().endswith(('.jpg', '.png'))][:max_imgs]
    with torch.no_grad():
        for f in files:
            img  = Image.open(os.path.join(folder, f)).convert('RGB')
            feat = backbone(transform(img).unsqueeze(0).to(DEVICE))
            features.append(feat.cpu().numpy()[0])
            labels.append(label)
    return features, labels

print("Extracting features...")
f1, l1 = extract_features_from_dir(REAL_FIELD_DIR,    "RSFD")
f2, l2 = extract_features_from_dir(REAL_MSLD_DIR, "Raw MSLD (Lab)")
f3, l3 = extract_features_from_dir(COMPOSITED_DIR,    "MSLD + Background")

all_features = np.array(f1 + f2 + f3)
all_labels   = l1 + l2 + l3

print("Running UMAP...")
coords = umap.UMAP(n_components=2, random_state=42).fit_transform(
    StandardScaler().fit_transform(all_features)
)

colors = {"RSFD": "green", "Raw MSLD (Lab)": "red", "MSLD + Background": "blue"}
plt.figure(figsize=(12, 8))
for label, color in colors.items():
    mask = [l == label for l in all_labels]
    pts  = coords[mask]
    plt.scatter(pts[:, 0], pts[:, 1], c=color, label=label, alpha=0.6, s=15)

plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend()
plt.tight_layout()
plt.savefig("composited_validation.png", dpi=150)
plt.show()

## Per-channel colour statistics

In [ ]:
import cv2
import numpy as np
import os

def get_channel_stats(folder, max_imgs=300):
    means, stds = [], []
    files = [f for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.png'))][:max_imgs]
    for f in files:
        img = cv2.imread(os.path.join(folder, f))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))
    return np.mean(means, axis=0), np.mean(stds, axis=0)

dirs = {
    "Real Field"    : REAL_FIELD_DIR,
    "Real MSLD" : REAL_MSLD_DIR,
    "Composited"    : COMPOSITED_DIR,
}

print(f"{'Group':<25} {'R mean':>8} {'G mean':>8} {'B mean':>8} {'R std':>8} {'G std':>8} {'B std':>8}")
print("-" * 75)
stats = {}
for name, folder in dirs.items():
    mean, std = get_channel_stats(folder)
    stats[name] = (mean, std)
    print(f"{name:<25} {mean[0]:>8.4f} {mean[1]:>8.4f} {mean[2]:>8.4f} {std[0]:>8.4f} {std[1]:>8.4f} {std[2]:>8.4f}")

print("\nDelta from Real Field:")
ref_mean, ref_std = stats["Real Field"]
for name in ["Real MSLD", "Composited"]:
    m, s = stats[name]
    print(f"  {name:<22} Mean Δ: {np.abs(m - ref_mean).mean():.4f} | Std Δ: {np.abs(s - ref_std).mean():.4f}")

## FID comparison

In [ ]:
import subprocess

result = subprocess.run([
    "python", "-m", "pytorch_fid",
    REAL_FIELD_DIR,
    COMPOSITED_DIR,
    "--device", "cuda"
], capture_output=True, text=True)

print(f"FID (Composited vs Real Field): {result.stdout.strip()}")

result2 = subprocess.run([
    "python", "-m", "pytorch_fid",
    REAL_FIELD_DIR,
    REAL_MSLD_DIR,
    "--device", "cuda"
], capture_output=True, text=True)

print(f"FID (Raw MSLD vs Real Field): {result2.stdout.strip()}")